# Login in hugginFace

In [1]:
from huggingface_hub import notebook_login

notebook_login()

# Import data

In [2]:
import pandas as pd

df = pd.read_csv('Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv')

df = df.reset_index(drop=True)
df

,url,title,text,authors,domain,score,country,language,topics,paywall,opinion_advocacy,label
0,https://nemontananews.com/stories/670532029-1-...,1 gas station in cities throughout Phillips Co...,Federal Energy Regulatory Commission (FERC) Ch...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
1,https://nemontananews.com/stories/670175501-mc...,McCone County: Income Maintenance dependency r...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
2,https://nemontananews.com/stories/670489322-da...,"Daniels County residents received $13,488 per ...","John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
3,https://nemontananews.com/stories/670175454-da...,Daniels County: Income Maintenance made up 0.8...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
4,https://nemontananews.com/stories/670413125-wh...,Where could drivers find the cheapest gas in c...,Federal Energy Regulatory Commission (FERC) Co...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...
19471,https://www.truthdig.com/articles/us-house-vot...,US House Vote Rolls Back Wetlands Regulations,The U.S. House voted Thursday to undo a Biden ...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19472,https://www.truthdig.com/articles/environmenta...,Environmental Racism in Canada,Informed nine months after the fact that its h...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19473,https://www.truthdig.com/articles/the-runaway-...,The Runaway Train of Corporate Greed,Thursday’s U.S. Senate hearing about the ongoi...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19474,https://www.truthdig.com/articles/ron-desantis...,Ron DeSantis’ Phantom War Against Voter Fraud,A new Florida law gives Gov.Ron DeSantis’ pros...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3


# BERT

In [3]:
import gc
import torch
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm
import os


def bert_cross_validation_with_domain_stratified_split(
    df: pd.DataFrame,
    text_column: str,
    label_column: str,
    domain_column: str,
    output_csv_path: str,
    model_name: str = "bert-base-uncased",
    n_splits: int = 5,
    seed: int = 42,
    epochs: int = 5,
    max_length: int = 256,
    batch_size_train: int = 32,
    batch_size_eval: int = 16,
    learning_rate: float = 2e-5,
):
    """
    Performs k-fold cross-validation with a BERT-based model on a text classification
    task, ensuring that a domain is entirely in either the train or test set for
    each fold, while stratifying based on the most common label per domain.
    """
    # Set random seeds
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Map labels to integers
    unique_labels = sorted(df[label_column].unique())
    num_labels = len(unique_labels)
    label_map = {label: i for i, label in enumerate(unique_labels)}
    df[label_column] = df[label_column].map(label_map)
    print(f"Number of unique labels: {num_labels}, Label mapping: {label_map}")

    # Metrics storage
    balanced_accuracy_scores = []
    accuracy_scores = []
    precision_micro_scores, precision_macro_scores, precision_weighted_scores = [], [], []
    recall_micro_scores, recall_macro_scores, recall_weighted_scores = [], [], []
    f1_micro_scores, f1_macro_scores, f1_weighted_scores = [], [], []
    confusions = []

    # Prepare domain-level data for stratified splitting
    grouped = df.groupby(domain_column)
    domains = list(grouped.groups.keys())
    # Use most frequent label in each domain for stratification
    domain_labels = [group[label_column].mode()[0] for _, group in grouped]

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = []

    # Create folds
    for train_idx, test_idx in skf.split(domains, domain_labels):
        train_domains = [domains[i] for i in train_idx]
        test_domains = [domains[i] for i in test_idx]
        train = df[df[domain_column].isin(train_domains)].reset_index(drop=True)
        test = df[df[domain_column].isin(test_domains)].reset_index(drop=True)
        folds.append((train, test))

    # Process each fold
    for fold_idx, (train, test) in enumerate(folds):
        print(f"\n{'='*40}")
        print(f"Processing BERT Fold {fold_idx+1}/{n_splits}")
        print(f"{'='*40}")

        # Print unique domains
        print(f"Train Domains: {train[domain_column].unique()}")
        print(f"Test Domains: {test[domain_column].unique()}")

        # Print label distribution percentages
        train_dist = train[label_column].value_counts(normalize=True) * 100
        test_dist = test[label_column].value_counts(normalize=True) * 100
        print("Train label distribution (%):")
        print(train_dist)
        print("Test label distribution (%):")
        print(test_dist)

        # Cleanup
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Tokenization
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        train_encodings = tokenizer(
            train[text_column].tolist(),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )
        test_encodings = tokenizer(
            test[text_column].tolist(),
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt",
        )

        # Datasets & Dataloaders
        train_dataset = TensorDataset(
            train_encodings.input_ids,
            train_encodings.attention_mask,
            torch.tensor(train[label_column].tolist())
        )
        test_dataset = TensorDataset(
            test_encodings.input_ids,
            test_encodings.attention_mask,
            torch.tensor(test[label_column].tolist())
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size_train, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size_eval)

        # Model
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=num_labels
        ).to(device)
        optimizer = AdamW(model.parameters(), lr=learning_rate)

        # Training
        for epoch in range(epochs):
            model.train()
            total_loss = 0
            progress_bar = tqdm(train_loader, desc=f"Fold {fold_idx+1} Epoch {epoch+1}/{epochs}")
            for batch in progress_bar:
                input_ids, attention_mask, labels = [b.to(device) for b in batch]
                optimizer.zero_grad()
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
                progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            print(f"Epoch {epoch+1} avg loss: {total_loss/len(train_loader):.4f}")

        # Evaluation
        model.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"Evaluating Fold {fold_idx+1}"):
                input_ids, attention_mask, labels = batch
                input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
                outputs = model(input_ids, attention_mask=attention_mask)
                preds = torch.argmax(outputs.logits, dim=1).cpu()
                val_preds.extend(preds.tolist())
                val_labels.extend(labels.tolist())

        # Metrics
        balanced_acc = balanced_accuracy_score(val_labels, val_preds)
        acc = accuracy_score(val_labels, val_preds)
        precision_micro = precision_score(val_labels, val_preds, average="micro")
        precision_macro = precision_score(val_labels, val_preds, average="macro", zero_division=0)
        precision_weighted = precision_score(val_labels, val_preds, average="weighted", zero_division=0)
        recall_micro = recall_score(val_labels, val_preds, average="micro")
        recall_macro = recall_score(val_labels, val_preds, average="macro", zero_division=0)
        recall_weighted = recall_score(val_labels, val_preds, average="weighted", zero_division=0)
        f1_micro = f1_score(val_labels, val_preds, average="micro")
        f1_macro = f1_score(val_labels, val_preds, average="macro", zero_division=0)
        f1_weighted = f1_score(val_labels, val_preds, average="weighted", zero_division=0)
        confusion = confusion_matrix(val_labels, val_preds).tolist()

        # Store metrics
        balanced_accuracy_scores.append(balanced_acc)
        accuracy_scores.append(acc)
        precision_micro_scores.append(precision_micro)
        precision_macro_scores.append(precision_macro)
        precision_weighted_scores.append(precision_weighted)
        recall_micro_scores.append(recall_micro)
        recall_macro_scores.append(recall_macro)
        recall_weighted_scores.append(recall_weighted)
        f1_micro_scores.append(f1_micro)
        f1_macro_scores.append(f1_macro)
        f1_weighted_scores.append(f1_weighted)
        confusions.append(confusion)

        # Save fold results
        fold_results = pd.DataFrame({
            "Fold": [fold_idx + 1],
            "Balanced_Accuracy": [balanced_acc],
            "Accuracy": [acc],
            "Precision_Micro": [precision_micro],
            "Precision_Macro": [precision_macro],
            "Precision_Weighted": [precision_weighted],
            "Recall_Micro": [recall_micro],
            "Recall_Macro": [recall_macro],
            "Recall_Weighted": [recall_weighted],
            "F1_Micro": [f1_micro],
            "F1_Macro": [f1_macro],
            "F1_Weighted": [f1_weighted],
            "Confusion_Matrix": [str(confusion)],
            "True_Labels": [str(val_labels)],
            "Predicted_Labels": [str(val_preds)],
        })


        # Inside your cross-validation loop, after training & evaluation:
        model_save_dir = 'Results/Classification/' + label_column + '/Models'
        os.makedirs(model_save_dir, exist_ok=True)
        
        # Define a unique name for each fold model
        model_save_path = os.path.join(model_save_dir, f"bert_fold_{fold_idx+1}.pt")
        
        # Save model state_dict
        torch.save(model.state_dict(), model_save_path)
        print(f"Fold {fold_idx+1} model saved to {model_save_path}")
        
        # Continue with saving results
        try:
            existing_results = pd.read_csv(output_csv_path)
            fold_results = pd.concat([existing_results, fold_results], ignore_index=True)
        except FileNotFoundError:
            pass
        fold_results.to_csv(output_csv_path, index=False)
        print(f"Fold {fold_idx+1} results saved to {output_csv_path}")
        
        # Cleanup
        del model, optimizer, train_loader, test_loader
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    print('Cross validation done!')

/home/fabio.pinelli/anaconda3/lib/python3.11/site-packages/torch/cuda/__init__.py:628: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [ ]:
bert_cross_validation_with_domain_stratified_split(
    df=df,
    text_column="text",
    label_column="label",
    output_csv_path="Results/Classification/bert.csv",
    domain_column="domain",
)


Number of unique labels: 5, Label mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}

Processing BERT Fold 1/5
Train Domains: ['nemontananews.com' 'theguardiansofdemocracy.com'
 'middlesborodailynews.com' 'tribunemag.co.uk' 'valiantnews.com'
 'realclearinvestigations.com' 'infacts.org' 'patriotsgazette.com'
 'salisburypost.com' 'riverfronttimes.com' 'bluestatebluesnews.com'
 'politico.eu' 'ncindiananews.com' 'anthonyblogan.com'
 'arizonadailyindependent.com' 'aei.org' 'texastribune.org'
 'southtulsatoday.com' 'indystandard.com' 'defenseone.com'
 'eciowanews.com' 'thecoastlandtimes.com' 'southwinstonsalemnews.com'
 'clantonadvertiser.com' 'dcenquirer.com' 'thedcpatriot.com'
 'indo-caribbean.com' 'state-journal.com' 'spiked-online.com'
 'gatewayreporter.com' 'africanvoiceonline.co.uk' 'conservtribune.com'
 'foreignpolicy.com' 'dailysoundandfury.com' 'brewtonstandard.com'
 'taz.de' 'unioncountyreview.com' 'arabnews.com' 'sfbayview.com'
 'theamericanconservative.com' 'ijr.org' 'muslimnews.co.uk'
 'fo

2025-11-11 09:26:00.227767: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-11 09:26:00.387034: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762853160.450388  398525 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762853160.469710  398525 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762853160.593377  398525 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Epoch 1 avg loss: 0.8655


Fold 1 Epoch 2/5: 100%|██████████| 487/487 [03:27<00:00,  2.34it/s, loss=0.3171]


Epoch 2 avg loss: 0.5281


...

Fold 4 Epoch 5/5: 100%|██████████| 488/488 [03:26<00:00,  2.36it/s, loss=0.0289]


Epoch 5 avg loss: 0.1154


Evaluating Fold 4: 100%|██████████████████████| 242/242 [00:16<00:00, 14.72it/s]


Fold 4 model saved to Results/Classification/label/Models/bert_fold_4.pt
Fold 4 results saved to Results/Classification/bert.csv

Processing BERT Fold 5/5
Train Domains: ['middlesborodailynews.com' 'ecoklahomanews.com'
 'realclearinvestigations.com' 'infacts.org' 'westhamiltonnews.com'
 'patriotsgazette.com' 'salisburypost.com' 'bluestatebluesnews.com'
 'themadisonrecord.com' 'politico.eu' 'ncindiananews.com'
 'anthonyblogan.com' 'arizonadailyindependent.com' 'aei.org'
 'texastribune.org' 'southtulsatoday.com' 'nationalmemo.com'
 'indystandard.com' 'eciowanews.com' 'southwinstonsalemnews.com'
 'dcenquirer.com' 'thedcpatriot.com' 'indo-caribbean.com'
 'state-journal.com' 'spiked-online.com' 'gatewayreporter.com' 'ntd.com'
 'africanvoiceonline.co.uk' 'foreignpolicy.com' 'taz.de'
 'unioncountyreview.com' 'arabnews.com' 'sfbayview.com'
 'swmontanatimes.com' 'theamericanconservative.com' 'ijr.org'
 'muslimnews.co.uk' 'foothillsreview.com' 'coppercourier.com'
 'christianpost.com' 'providence

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fold 5 Epoch 1/5:   0%|                                 | 0/484 [00:00<?, ?it/s]/home/fabio.pinelli/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1520: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Fold 5 Epoch 1/5:  63%|██████▎   | 305/484 [02:08<01:15,  2.36it/s, loss=0.5496]

# Memory cleanup

In [ ]:
import gc
import torch

# Manually remove large objects from previous run
llama_objects = ['model', 'tokenizer', 'train_loader', 'test_loader', 
                'train_dataset', 'test_dataset', 'train_encodings',
                'test_encodings', 'optimizer', 'scaler']

for obj in llama_objects:
    if obj in globals():
        del globals()[obj]

# Force garbage collection
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared: {} MB".format(torch.cuda.memory_allocated() // 1024**2))

In [1]:
import pandas as pd

df = pd.read_csv('Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv')

df = df.reset_index(drop=True)
df

,url,title,text,authors,domain,score,country,language,topics,paywall,opinion_advocacy,label
0,https://nemontananews.com/stories/670532029-1-...,1 gas station in cities throughout Phillips Co...,Federal Energy Regulatory Commission (FERC) Ch...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
1,https://nemontananews.com/stories/670175501-mc...,McCone County: Income Maintenance dependency r...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
2,https://nemontananews.com/stories/670489322-da...,"Daniels County residents received $13,488 per ...","John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
3,https://nemontananews.com/stories/670175454-da...,Daniels County: Income Maintenance made up 0.8...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
4,https://nemontananews.com/stories/670413125-wh...,Where could drivers find the cheapest gas in c...,Federal Energy Regulatory Commission (FERC) Co...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...
19471,https://www.truthdig.com/articles/us-house-vot...,US House Vote Rolls Back Wetlands Regulations,The U.S. House voted Thursday to undo a Biden ...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19472,https://www.truthdig.com/articles/environmenta...,Environmental Racism in Canada,Informed nine months after the fact that its h...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19473,https://www.truthdig.com/articles/the-runaway-...,The Runaway Train of Corporate Greed,Thursday’s U.S. Senate hearing about the ongoi...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19474,https://www.truthdig.com/articles/ron-desantis...,Ron DeSantis’ Phantom War Against Voter Fraud,A new Florida law gives Gov.Ron DeSantis’ pros...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3


# LLAMA3

In [ ]:
import random
import gc
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from tqdm import tqdm
import numpy as np
import pandas as pd
import torch
import os
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    BitsAndBytesConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
import bitsandbytes as bnb
from torch.optim import AdamW


# Set random seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def run_llama_finetuning_domain_stratified_cv(
    df,
    label_col,
    text_col,
    output_csv_path,
    domain_col=None,
    num_epochs=5,
    train_batch_size=32,
    eval_batch_size=64,
    gradient_accumulation_steps=2,
    max_seq_length=256,
    n_splits=5,
    model_id="meta-llama/Meta-Llama-3-8B",
    device=None,
    val_size=0.1,
    early_stopping_patience=2,
):
    """
    Improved fine-tuning flow with:
      - longer sequence length (default 256)
      - extended LoRA target modules
      - gradient checkpointing
      - per-fold train/val split and early stopping (based on val loss)
      - class-weighted loss to handle imbalance
      - LR scheduler (linear warmup + cosine decay)
      - dynamic eval batch fallback on OOM
      - proper model reinitialization per fold for correct CV
      - [NEW] Logs train and val loss per epoch to the output CSV.

    Note: this is still a memory-heavy recipe; adjust batch sizes or switch to smaller model if GPU OOM.
    """

    # --- Configuration & device ---
    DEVICE = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
    MODEL_ID = model_id
    NUM_EPOCHS = num_epochs
    TRAIN_BATCH_SIZE = train_batch_size
    EVAL_BATCH_SIZE = eval_batch_size
    GRADIENT_ACCUMULATION_STEPS = gradient_accumulation_steps
    MAX_SEQ_LENGTH = max_seq_length
    N_SPLITS = n_splits

    # --- Data and Label Processing ---
    df = df.copy().reset_index(drop=True)

    unique_labels = sorted(df[label_col].unique())
    num_labels = len(unique_labels)
    print(f"Detected {num_labels} unique labels: {unique_labels}")
    label_mapping = {label: i for i, label in enumerate(unique_labels)}
    df["mapped_label"] = df[label_col].map(label_mapping)
    print(f"Mapped labels: {label_mapping}")

    # --- Tokenizer (initialized once) ---
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    # ensure pad token
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # --- Quantization Config (reused for each fold) ---
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    # --- LoRA configuration (reused for each fold) ---
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "down_proj",
            "up_proj",
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="SEQ_CLS",
    )

    # --- Cross-Validation Folds ---
    if domain_col and domain_col in df.columns:
        print(f"Using domain-level stratification on column '{domain_col}'...")
        grouped = df.groupby(domain_col)
        domains = list(grouped.groups.keys())
        stratify_labels = [group[1]["mapped_label"].mode()[0] for group in grouped]
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
        folds = []
        for train_idx, test_idx in skf.split(domains, stratify_labels):
            train_domains = [domains[i] for i in train_idx]
            test_domains = [domains[i] for i in test_idx]
            train_df = df[df[domain_col].isin(train_domains)].reset_index(drop=True)
            test_df = df[df[domain_col].isin(test_domains)].reset_index(drop=True)
            folds.append((train_df, test_df))
    else:
        print("Using standard stratified splitting...")
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
        folds = [(df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True))
                 for train_idx, test_idx in skf.split(df, df["mapped_label"])]

    all_fold_results = []
    all_fold_preds = []

    # --- Training and Evaluation per fold ---
    for fold_idx, (train_df, test_df) in enumerate(folds):
        print(f"\n{'='*40}")
        print(f"Processing Fold {fold_idx+1}/{N_SPLITS}")
        print(f"{'='*40}")

        # split train into train/val for early stopping
        if len(train_df) > 1:
            stratify = train_df["mapped_label"] if len(train_df["mapped_label"].unique()) > 1 else None
            train_part, val_part = train_test_split(
                train_df,
                test_size=val_size,
                random_state=RANDOM_SEED,
                stratify=stratify,
            )
        else:
            train_part = train_df
            val_part = test_df.sample(frac=min(0.1, 1.0), random_state=RANDOM_SEED)

        # --- Tokenization ---
        train_encodings = tokenizer(
            train_part[text_col].tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_SEQ_LENGTH,
            return_tensors="pt",
        )
        val_encodings = tokenizer(
            val_part[text_col].tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_SEQ_LENGTH,
            return_tensors="pt",
        )
        test_encodings = tokenizer(
            test_df[text_col].tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_SEQ_LENGTH,
            return_tensors="pt",
        )

        train_dataset = TensorDataset(
            train_encodings.input_ids,
            train_encodings.attention_mask,
            torch.tensor(train_part["mapped_label"].tolist()),
        )
        val_dataset = TensorDataset(
            val_encodings.input_ids,
            val_encodings.attention_mask,
            torch.tensor(val_part["mapped_label"].tolist()),
        )
        test_dataset = TensorDataset(
            test_encodings.input_ids,
            test_encodings.attention_mask,
            torch.tensor(test_df["mapped_label"].tolist()),
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=TRAIN_BATCH_SIZE,
            shuffle=True,
            num_workers=2,
            pin_memory=True,
        )

        # --- CREATE FRESH MODEL FOR EACH FOLD ---
        print(f"Initializing fresh model for fold {fold_idx+1}...")
        base_model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID,
            quantization_config=quantization_config,
            num_labels=num_labels,
            device_map="auto",
        )
        base_model.config.pad_token_id = tokenizer.eos_token_id

        model = prepare_model_for_kbit_training(base_model)
        model = get_peft_model(model, lora_config)
        model.to(DEVICE)

        # enable gradient checkpointing to reduce memory at cost of compute
        try:
            model.gradient_checkpointing_enable()
        except Exception:
            pass

        # compute class weights for imbalanced data
        class_counts = train_part["mapped_label"].value_counts().sort_index()
        weights = 1.0 / (class_counts.values + 1e-12)
        weights = weights / weights.sum()
        class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)
        criterion = torch.nn.CrossEntropyLoss(weight=class_weights)

        # optimizer (8-bit Adam from bitsandbytes) — keep params in 8bit for memory
        try:
            optimizer = bnb.optim.AdamW8bit(model.parameters(), lr=2e-5)
        except Exception:
            optimizer = AdamW(model.parameters(), lr=2e-5)

        # scheduler: linear warmup + cosine decay
        num_update_steps_per_epoch = max(1, len(train_loader) // GRADIENT_ACCUMULATION_STEPS)
        max_train_steps = NUM_EPOCHS * num_update_steps_per_epoch
        num_warmup_steps = int(0.06 * max_train_steps)
        scheduler = get_scheduler(
            name="cosine",
            optimizer=optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=max_train_steps,
        )

        scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

        best_val_loss = float("inf")
        epochs_without_improve = 0
        best_model_state = None
        
        # <--- MODIFICA 1: Inizializza le liste per le loss del fold ---
        fold_train_losses = []
        fold_val_losses = []

        # --- Training ---
        for epoch in range(NUM_EPOCHS):
            model.train()
            total_loss = 0.0
            optimizer.zero_grad()
            pbar = tqdm(train_loader, desc=f"Fold {fold_idx+1} | Epoch {epoch+1}/{NUM_EPOCHS} (Training)")
            for step, batch in enumerate(pbar):
                input_ids = batch[0].to(DEVICE, non_blocking=True)
                attention_mask = batch[1].to(DEVICE, non_blocking=True)
                labels = batch[2].to(DEVICE, non_blocking=True)

                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                        logits = outputs.logits
                        loss = criterion(logits, labels) / GRADIENT_ACCUMULATION_STEPS
                    scaler.scale(loss).backward()
                else:
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    logits = outputs.logits
                    loss = criterion(logits, labels) / GRADIENT_ACCUMULATION_STEPS
                    loss.backward()

                if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                    if scaler is not None:
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                total_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
                pbar.set_postfix({"loss": total_loss / (step + 1)})
            
            pbar.close()
            # <--- MODIFICA 2a: Calcola e salva la loss media di training per l'epoca ---
            avg_train_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0.0
            fold_train_losses.append(avg_train_loss)

            # --- Validation after each epoch ---
            model.eval()
            val_losses = []
            val_preds = []
            val_trues = []
            val_loader = DataLoader(
                val_dataset,
                batch_size=EVAL_BATCH_SIZE,
                shuffle=False,
                num_workers=2,
                pin_memory=True,
            )
            with torch.no_grad():
                for batch in val_loader:
                    inputs = {
                        "input_ids": batch[0].to(DEVICE, non_blocking=True),
                        "attention_mask": batch[1].to(DEVICE, non_blocking=True),
                    }
                    labels = batch[2].to(DEVICE, non_blocking=True)
                    outputs = model(**inputs)
                    logits = outputs.logits
                    loss = criterion(logits, labels)
                    val_losses.append(loss.item())
                    preds = torch.argmax(logits, dim=1).cpu().numpy()
                    val_preds.extend(preds.tolist())
                    val_trues.extend(labels.cpu().numpy().tolist())

            mean_val_loss = float(np.mean(val_losses)) if len(val_losses) > 0 else float("inf")
            
            # <--- MODIFICA 2b: Salva la loss media di validazione per l'epoca ---
            fold_val_losses.append(mean_val_loss)
            
            # <--- MODIFICA 2c: Aggiorna il print per mostrare entrambe le loss ---
            print(f"Fold {fold_idx+1} Epoch {epoch+1} | train_loss={avg_train_loss:.4f} | val_loss={mean_val_loss:.4f}")

            # early stopping check
            if mean_val_loss < best_val_loss:
                best_val_loss = mean_val_loss
                epochs_without_improve = 0
                # save best model state in memory
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                print(f"New best model for fold {fold_idx+1} at epoch {epoch+1}")
            else:
                epochs_without_improve += 1
                if epochs_without_improve >= early_stopping_patience:
                    print(f"Early stopping at epoch {epoch+1} for fold {fold_idx+1}")
                    break

        # --- Load best model state for evaluation ---
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            model.to(DEVICE)
            print(f"Loaded best model state for fold {fold_idx+1}")

        # --- Evaluation on test set ---
        model.eval()
        val_preds_list = []
        val_true_labels = []

        eval_batch_size = EVAL_BATCH_SIZE
        while True:
            try:
                test_loader = DataLoader(
                    test_dataset,
                    batch_size=eval_batch_size,
                    shuffle=False,
                    num_workers=2,
                    pin_memory=True,
                )
                with torch.no_grad():
                    pbar = tqdm(test_loader, desc=f"Fold {fold_idx+1} (Evaluating)")
                    for batch in pbar:
                        inputs = {
                            "input_ids": batch[0].to(DEVICE, non_blocking=True),
                            "attention_mask": batch[1].to(DEVICE, non_blocking=True),
                        }
                        outputs = model(**inputs)
                        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                        val_preds_list.extend(preds.tolist())
                        val_true_labels.extend(batch[2].cpu().numpy().tolist())
                break
            except RuntimeError as e:
                if "out of memory" in str(e):
                    torch.cuda.empty_cache()
                    eval_batch_size = max(1, eval_batch_size // 2)
                    print(f"[OOM] Reducing eval batch size to {eval_batch_size} and retrying...")
                else:
                    raise

        # --- Metrics ---
        balanced_acc = balanced_accuracy_score(val_true_labels, val_preds_list)
        accuracy = accuracy_score(val_true_labels, val_preds_list)
        precision_micro = precision_score(val_true_labels, val_preds_list, average="micro", zero_division=0)
        precision_macro = precision_score(val_true_labels, val_preds_list, average="macro", zero_division=0)
        precision_weighted = precision_score(val_true_labels, val_preds_list, average="weighted", zero_division=0)
        recall_micro = recall_score(val_true_labels, val_preds_list, average="micro", zero_division=0)
        recall_macro = recall_score(val_true_labels, val_preds_list, average="macro", zero_division=0)
        recall_weighted = recall_score(val_true_labels, val_preds_list, average="weighted", zero_division=0)
        f1_micro = f1_score(val_true_labels, val_preds_list, average="micro", zero_division=0)
        f1_macro = f1_score(val_true_labels, val_preds_list, average="macro", zero_division=0)
        f1_weighted = f1_score(val_true_labels, val_preds_list, average="weighted", zero_division=0)
        confusion = confusion_matrix(val_true_labels, val_preds_list).tolist()

        # <--- MODIFICA 3: Aggiungi le liste di loss al dizionario dei risultati ---
        fold_results = {
            "Fold": fold_idx + 1,
            "Train_Losses_Per_Epoch": str(fold_train_losses),  # <--- Aggiunto
            "Val_Losses_Per_Epoch": str(fold_val_losses),      # <--- Aggiunto
            "Balanced_Accuracy": balanced_acc,
            "Accuracy": accuracy,
            "Precision_Micro": precision_micro,
            "Precision_Macro": precision_macro,
            "Precision_Weighted": precision_weighted,
            "Recall_Micro": recall_micro,
            "Recall_Macro": recall_macro,
            "Recall_Weighted": recall_weighted,
            "F1_Micro": f1_micro,
            "F1_Macro": f1_macro,
            "F1_Weighted": f1_weighted,
            "Confusion_Matrix": str(confusion),
            "Val_True_Labels": str(val_true_labels),
            "Val_Predictions": str(val_preds_list),
        }

        all_fold_results.append(fold_results)
        all_fold_preds.append(val_preds_list)

        # Save model for this fold
        model_save_dir = f"Results/Classification/{label_col}/Models"
        os.makedirs(model_save_dir, exist_ok=True)
        model_save_path = os.path.join(model_save_dir, f"llama_fold_{fold_idx+1}.pt")
        try:
            torch.save(model.state_dict(), model_save_path)
            print(f"Fold {fold_idx+1} model saved to {model_save_path}")
        except Exception as e:
            print(f"Could not save model for fold {fold_idx+1}: {e}")

        # Cleanup
        del model, optimizer, scheduler, train_loader, test_loader, val_loader
        del base_model, best_model_state
        torch.cuda.empty_cache()
        gc.collect()

        # Save results after each fold (overwrites the CSV)
        results_df = pd.DataFrame(all_fold_results)
        os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
        results_df.to_csv(output_csv_path, index=False)
        print(f"Results after fold {fold_idx+1} saved to {output_csv_path}")

    print("\nCross-validation completed. Final results saved to", output_csv_path)
    return results_df

/home/fabio.pinelli/anaconda3/lib/python3.11/site-packages/torch/cuda/__init__.py:628: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
2025-11-12 09:42:21.402148: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-12 09:42:21.447430: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762940541.500786 1047220 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762940541.519912 1047220 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has alre

In [ ]:
results_df = run_llama_finetuning_domain_stratified_cv(
    df=df,
    text_col="text",
    label_col="label",
    output_csv_path="Results/Classification/llama.csv",
    domain_col="domain"
)


Detected 5 unique labels: [0, 1, 2, 3, 4]
Mapped labels: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}
Using domain-level stratification on column 'domain'...

Processing Fold 1/5
Initializing fresh model for fold 1...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Meta-Llama-3-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fold 1 | Epoch 1/5 (Training):   0%|                    | 0/439 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | 

# Memory cleanup

In [ ]:
import gc
import torch

# Manually remove large objects from previous run
llama_objects = ['model', 'tokenizer', 'train_loader', 'test_loader', 
                'train_dataset', 'test_dataset', 'train_encodings',
                'test_encodings', 'optimizer', 'scaler']

for obj in llama_objects:
    if obj in globals():
        del globals()[obj]

# Force garbage collection
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared: {} MB".format(torch.cuda.memory_allocated() // 1024**2))

In [2]:
import pandas as pd

df = pd.read_csv('Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv')

df = df.reset_index(drop=True)
df

,url,title,text,authors,domain,score,country,language,topics,paywall,opinion_advocacy,label
0,https://nemontananews.com/stories/670532029-1-...,1 gas station in cities throughout Phillips Co...,Federal Energy Regulatory Commission (FERC) Ch...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
1,https://nemontananews.com/stories/670175501-mc...,McCone County: Income Maintenance dependency r...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
2,https://nemontananews.com/stories/670489322-da...,"Daniels County residents received $13,488 per ...","John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
3,https://nemontananews.com/stories/670175454-da...,Daniels County: Income Maintenance made up 0.8...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
4,https://nemontananews.com/stories/670413125-wh...,Where could drivers find the cheapest gas in c...,Federal Energy Regulatory Commission (FERC) Co...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,No,Yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...
19471,https://www.truthdig.com/articles/us-house-vot...,US House Vote Rolls Back Wetlands Regulations,The U.S. House voted Thursday to undo a Biden ...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19472,https://www.truthdig.com/articles/environmenta...,Environmental Racism in Canada,Informed nine months after the fact that its h...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19473,https://www.truthdig.com/articles/the-runaway-...,The Runaway Train of Corporate Greed,Thursday’s U.S. Senate hearing about the ongoi...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3
19474,https://www.truthdig.com/articles/ron-desantis...,Ron DeSantis’ Phantom War Against Voter Fraud,A new Florida law gives Gov.Ron DeSantis’ pros...,NaN,truthdig.com,80.0,NaN,NaN,Political_news_or_commentary,NaN,NaN,3


# Roberta

In [5]:
import gc
import torch
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import os
import warnings
warnings.filterwarnings('ignore')


def set_deterministic_seed(seed=42):
    """
    Configura il seed in modo completamente deterministico per riproducibilità.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # Importante: questi settings garantiscono determinismo ma riducono leggermente le performance
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Per trasformers
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    
    # PyTorch 1.8+: abilita modalità deterministica
    if hasattr(torch, 'use_deterministic_algorithms'):
        torch.use_deterministic_algorithms(True, warn_only=True)


def roberta_cross_validation_domain_stratified(
    df, 
    label_col, 
    text_col, 
    results_csv_path, 
    domain_col=None, 
    seed=42,
    batch_size=32,
    num_epochs=5,
    learning_rate=2e-5,
    max_length=256,
    gradient_accumulation_steps=1,
    use_deterministic=True
):
    """
    Performs 5-fold domain-level stratified cross-validation using RoBERTa-large.
    
    Miglioramenti principali:
    - Determinismo completo se use_deterministic=True
    - Gradient accumulation per batch effettivi più grandi
    - Ottimizzazioni memoria con chunked processing
    - Early stopping opzionale
    - Migliore gestione mixed precision
    - DataLoader più efficiente
    """
    
    # Configura determinismo
    if use_deterministic:
        set_deterministic_seed(seed)
        print(f"✓ Modalità deterministica abilitata con seed={seed}")
    else:
        # Performance mode
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = False
            torch.backends.cudnn.benchmark = True
        print(f"✓ Modalità performance abilitata con seed={seed}")

    model_name = "roberta-large"
    df = df.reset_index(drop=True).copy()
    
    # Label mapping
    unique_labels = sorted(df[label_col].unique())
    label_map = {label: i for i, label in enumerate(unique_labels)}
    df['mapped_label'] = df[label_col].map(label_map)
    num_labels = len(unique_labels)

    print(f"\n{'='*60}")
    print(f"Label mapping: {label_map}")
    print(f"Detected {num_labels} unique labels")
    print(f"{'='*60}\n")

    # Device setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    # --- TOKENIZZAZIONE OTTIMIZZATA ---
    print("\n🔄 Pre-tokenizing all data...")
    
    # Tokenizza in chunks per evitare OOM su dataset grandi
    chunk_size = 1000
    all_input_ids = []
    all_attention_masks = []
    
    for i in tqdm(range(0, len(df), chunk_size), desc="Tokenizing"):
        chunk = df[text_col].iloc[i:i+chunk_size].tolist()
        encodings = tokenizer(
            chunk,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="pt"
        )
        all_input_ids.append(encodings.input_ids)
        all_attention_masks.append(encodings.attention_mask)
    
    # Concatena tutto
    all_input_ids = torch.cat(all_input_ids, dim=0).contiguous()
    all_attention_masks = torch.cat(all_attention_masks, dim=0).contiguous()
    
    print(f"✓ Tokenization complete. Shape: {all_input_ids.shape}")
    
    # Mantieni su CPU per ora, spostiamo su GPU solo i batch necessari
    # Questo risparmia memoria GPU per dataset grandi
    all_labels = torch.tensor(df['mapped_label'].tolist(), dtype=torch.long)

    # --- DOMAIN-STRATIFIED FOLDS ---
    print("\n📊 Creating stratified folds...")
    if domain_col and domain_col in df.columns:
        grouped = df.groupby(domain_col)
        domains = list(grouped.groups.keys())
        stratify_labels = [group['mapped_label'].mode()[0] for _, group in grouped]
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        folds = []
        for train_idx, test_idx in skf.split(domains, stratify_labels):
            train_domains = [domains[i] for i in train_idx]
            test_domains = [domains[i] for i in test_idx]
            train_indices = df[df[domain_col].isin(train_domains)].index.tolist()
            test_indices = df[df[domain_col].isin(test_domains)].index.tolist()
            folds.append((train_indices, test_indices))
        print(f"✓ Domain-stratified folds created")
    else:
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        folds = [(train_idx.tolist(), test_idx.tolist()) 
                 for train_idx, test_idx in skf.split(df, df['mapped_label'])]
        print(f"✓ Standard stratified folds created")

    all_fold_results = []

    # --- TRAINING LOOP ---
    for fold_idx, (train_indices, test_indices) in enumerate(folds):
        print(f"\n{'='*60}")
        print(f"🚀 FOLD {fold_idx+1}/5")
        print(f"{'='*60}")

        train = df.iloc[train_indices].reset_index(drop=True)
        test = df.iloc[test_indices].reset_index(drop=True)

        if domain_col and domain_col in df.columns:
            print(f"Train Domains ({len(train[domain_col].unique())}): {sorted(train[domain_col].unique())[:5]}...")
            print(f"Test Domains ({len(test[domain_col].unique())}): {sorted(test[domain_col].unique())[:5]}...")

        print(f"\nTrain size: {len(train)} | Test size: {len(test)}")
        print("\nTrain label distribution (%):")
        print(train['mapped_label'].value_counts(normalize=True).mul(100).round(2))
        print("\nTest label distribution (%):")
        print(test['mapped_label'].value_counts(normalize=True).mul(100).round(2))

        # Cleanup
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Create datasets
        train_dataset = TensorDataset(
            all_input_ids[train_indices],
            all_attention_masks[train_indices],
            all_labels[train_indices]
        )
        test_dataset = TensorDataset(
            all_input_ids[test_indices],
            all_attention_masks[test_indices],
            all_labels[test_indices]
        )

        # DataLoaders ottimizzati
        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=2,  # Usa workers per pre-fetching
            pin_memory=True,
            prefetch_factor=2,
            persistent_workers=True,
            generator=torch.Generator().manual_seed(seed) if use_deterministic else None
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=batch_size * 2,  # Batch più grandi per inference
            shuffle=False,
            num_workers=2,
            pin_memory=True,
            prefetch_factor=2,
            persistent_workers=True
        )

        # Load model
        print("\n🔧 Loading model...")
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels
        )
        
        # Gradient checkpointing per risparmiare memoria
        if hasattr(model, 'gradient_checkpointing_enable'):
            model.gradient_checkpointing_enable()
        
        model = model.to(device)

        # Optimizer con fused operations per performance
        optimizer = AdamW(
            model.parameters(),
            lr=learning_rate,
            weight_decay=0.01,
            eps=1e-8,
            fused=torch.cuda.is_available() and not use_deterministic
        )

        # Scheduler
        num_training_steps = len(train_loader) * num_epochs // gradient_accumulation_steps
        num_warmup_steps = num_training_steps // 10
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps
        )

        # Mixed precision setup
        use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported() and not use_deterministic
        dtype = torch.bfloat16 if use_bf16 else torch.float16
        scaler = None if use_bf16 else (GradScaler() if torch.cuda.is_available() else None)

        print(f"Mixed precision: {'bfloat16' if use_bf16 else 'float16' if scaler else 'disabled'}")
        print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
        print(f"Effective batch size: {batch_size * gradient_accumulation_steps}")

        # --- TRAINING ---
        print("\n🏋️ Training...")
        best_val_loss = float('inf')
        
        for epoch in range(num_epochs):
            model.train()
            train_loss = 0
            optimizer.zero_grad(set_to_none=True)
            
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
            
            for step, batch in enumerate(progress_bar):
                input_ids = batch[0].to(device, non_blocking=True)
                attention_mask = batch[1].to(device, non_blocking=True)
                labels = batch[2].to(device, non_blocking=True)

                # Mixed precision training
                if torch.cuda.is_available():
                    with autocast(dtype=dtype):
                        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                        loss = outputs.loss / gradient_accumulation_steps
                    
                    if scaler is not None:
                        scaler.scale(loss).backward()
                    else:
                        loss.backward()
                else:
                    outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                    loss = outputs.loss / gradient_accumulation_steps
                    loss.backward()

                # Update ogni gradient_accumulation_steps
                if (step + 1) % gradient_accumulation_steps == 0:
                    if scaler is not None:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                        optimizer.step()
                    
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

                train_loss += loss.item() * gradient_accumulation_steps
                progress_bar.set_postfix({'loss': f"{loss.item() * gradient_accumulation_steps:.4f}"})

            avg_loss = train_loss / len(train_loader)
            print(f"Epoch {epoch+1} - Avg Loss: {avg_loss:.4f}")

        # --- EVALUATION ---
        print("\n📈 Evaluating...")
        model.eval()
        val_preds, val_labels_list = [], []

        with torch.no_grad():
            for batch in tqdm(test_loader, desc="Evaluating"):
                input_ids = batch[0].to(device, non_blocking=True)
                attention_mask = batch[1].to(device, non_blocking=True)
                labels = batch[2]

                if torch.cuda.is_available():
                    with autocast(dtype=dtype):
                        outputs = model(input_ids, attention_mask=attention_mask)
                else:
                    outputs = model(input_ids, attention_mask=attention_mask)

                preds = torch.argmax(outputs.logits, dim=1)
                val_preds.extend(preds.cpu().tolist())
                val_labels_list.extend(labels.tolist())

        # Metrics
        balanced_acc = balanced_accuracy_score(val_labels_list, val_preds)
        accuracy = accuracy_score(val_labels_list, val_preds)
        precision_macro = precision_score(val_labels_list, val_preds, average="macro", zero_division=0)
        precision_weighted = precision_score(val_labels_list, val_preds, average="weighted", zero_division=0)
        recall_macro = recall_score(val_labels_list, val_preds, average="macro", zero_division=0)
        recall_weighted = recall_score(val_labels_list, val_preds, average="weighted", zero_division=0)
        f1_macro = f1_score(val_labels_list, val_preds, average="macro", zero_division=0)
        f1_weighted = f1_score(val_labels_list, val_preds, average="weighted", zero_division=0)
        confusion = confusion_matrix(val_labels_list, val_preds).tolist()

        print(f"\n📊 Results:")
        print(f"  Balanced Accuracy: {balanced_acc:.4f}")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  F1 Macro: {f1_macro:.4f}")
        print(f"  F1 Weighted: {f1_weighted:.4f}")

        fold_results = pd.DataFrame({
            "Fold": [fold_idx + 1],
            "Balanced_Accuracy": [balanced_acc],
            "Accuracy": [accuracy],
            "Precision_Macro": [precision_macro],
            "Precision_Weighted": [precision_weighted],
            "Recall_Macro": [recall_macro],
            "Recall_Weighted": [recall_weighted],
            "F1_Macro": [f1_macro],
            "F1_Weighted": [f1_weighted],
            "Confusion_Matrix": [confusion],
            "Val_True_Labels": [val_labels_list],
            "Val_Predictions": [val_preds]
        })

        # Save model
        model_save_dir = f"Results/Classification/{label_col}/Models"
        os.makedirs(model_save_dir, exist_ok=True)
        model_save_path = os.path.join(model_save_dir, f"roberta_fold_{fold_idx+1}.pt")
        
        torch.save(model.state_dict(), model_save_path)
        print(f"💾 Model saved to {model_save_path}")

        # Save results
        try:
            existing = pd.read_csv(results_csv_path)
            results = pd.concat([existing, fold_results], ignore_index=True)
        except FileNotFoundError:
            results = fold_results

        results.to_csv(results_csv_path, index=False)
        print(f"💾 Results saved to {results_csv_path}")

        # Cleanup
        del model, optimizer, scheduler, train_loader, test_loader
        del train_dataset, test_dataset
        if scaler is not None:
            del scaler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Final cleanup
    del all_input_ids, all_attention_masks, all_labels
    gc.collect()

    print(f"\n{'='*60}")
    print("✅ Cross-validation completed successfully!")
    print(f"{'='*60}\n")
    
    return results

In [ ]:
roberta_cross_validation_domain_stratified(
    df=df,
    text_col="text",
    label_col="label",
    results_csv_path="Results/Classification/roberta.csv",
    domain_col="domain",
)

✓ Modalità deterministica abilitata con seed=42

Label mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}
Detected 5 unique labels

Using device: cuda
GPU: Tesla V100S-PCIE-32GB
GPU Memory: 34.09 GB

🔄 Pre-tokenizing all data...


Tokenizing: 100%|███████████████████████████████| 20/20 [00:18<00:00,  1.08it/s]


✓ Tokenization complete. Shape: torch.Size([19476, 256])

📊 Creating stratified folds...
✓ Domain-stratified folds created

🚀 FOLD 1/5
Train Domains (351): ['5pillarsuk.com', 'aei.org', 'africanvoiceonline.co.uk', 'alabamanow.com', 'albertleatribune.com']...
Test Domains (88): ['aapsonline.org', 'aclu.org', 'algemeiner.com', 'americanwirenews.com', 'analyzingamerica.org']...

Train size: 15581 | Test size: 3895

Train label distribution (%):
mapped_label
0    25.38
3    25.21
2    24.78
1    14.25
4    10.38
Name: proportion, dtype: float64

Test label distribution (%):
mapped_label
3    28.58
0    23.54
1    20.44
2    19.49
4     7.96
Name: proportion, dtype: float64

🔧 Loading model...


2025-11-14 10:16:27.122838: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-14 10:16:27.144570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763115387.168905 2339933 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763115387.177055 2339933 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1763115387.200348 2339933 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Mixed precision: float16
Gradient accumulation steps: 1
Effective batch size: 32

🏋️ Training...


Epoch 1/5:   0%|                                        | 0/487 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Epoch 1/5:  23%|███▊             | 111/487 [01:01<03:31,  1.78it/s, loss=0.8436]